### Check for common Patient ID between 2 datasets
used for exploratory analysis on original datasets because there are some NA patient ids in 'source of truth'

In [ ]:
def find_common (df1, df2):
    common_ids = pd.Index(df1['Random ID'].dropna().unique()).intersectioBROn(
        pd.Index(df2['Random ID'].dropna().unique())
    )

    # If you want rows from each df where the ID is common:
    df1_common = df1[df1['Random ID'].isin(common_ids)]
    df2_common = df2[df2['Random ID'].isin(common_ids)]

    # Ensure the column is treated consistently (optional but recommended)
    df1['Random ID'] = df1['Random ID'].astype(str)
    df2['Random ID'] = df2['Random ID'].astype(str)
    return common_ids

In [ ]:
print(sorted(find_common(cleaned_outpatient, df2)))

### Check for missing Patient ID between 2 datasets

In [ ]:
def get_missing_ids(df_source, df_target, id_col='Random ID'):
    """
    Returns IDs present in df_source but missing from df_target.
    """
    source_ids = set(df_source[id_col])
    target_ids = set(df_target[id_col])
    
    missing = list(source_ids - target_ids)
    return missing

# Usage:
missing_ids = get_missing_ids(cleaned_outpatient, df2)
missing_ids2 = get_missing_ids(df2,cleaned_outpatient)
print(missing_ids2)

## Export FN and FP Sources
(if the code below doesnt work, rerun the first cell of this notebook agn)

In [ ]:
import pandas as pd
df_imaging=pd.read_csv('cleaned_radio.csv')
df_labs=pd.read_csv('cleaned_lab.csv')
df_truth=pd.read_csv('cleaned_truth.csv')
df_discharge=pd.read_csv('cleaned_discharge_summary.csv')
df_outpatient=pd.read_csv('cleaned_outpatient_summary.csv')
df_endoscope=pd.read_csv('cleaned_endoscope.csv')

In [ ]:
df_patients=(df_truth.copy()[['Random ID']]).dropna(subset=['Random ID'])
df_patients

import pandas as pd

def process_labs(df_labs):
    """Clean, pivot, and flatten lab data."""
    df = df_labs.copy()
    # Parse date safely
    df['timestamp'] = pd.to_datetime(df['Reported Date'], errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort to preserve chronological order in concatenation
    df = df.sort_values(['Random ID', 'timestamp'])

    # Create the string timeline for the LLM
    df['Lab_Entry'] = (
        df['timestamp'].dt.strftime('%Y-%m-%d') + " " +
        df['Lab Resulted Order Test Description'].astype(str) + ": " +
        df['Result Value'].astype(str)
    )

    # Group into a single string per patient
    return df.groupby('Random ID', sort=False)['Lab_Entry'].apply(' | '.join).reset_index()
def process_imaging(df_imaging):
    df = df_imaging.copy()
    # Remove dayfirst=True to let pandas handle the YYYY-MM-DD format correctly
    df['timestamp'] = pd.to_datetime(df['Performed Date Time'], errors='coerce')
    df = df.dropna(subset=['timestamp'])

    df = df.sort_values(['Random ID', 'timestamp'])
    df['Img_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Text'].astype(str)

    return df.groupby('Random ID', sort=False)['Img_Entry'].apply(' | '.join).reset_index()

def process_discharge(df_discharge):
    """
    Clean and flatten discharge summaries.

    Expected columns:
      - 'Visit Date (YYYYMMDD)' : date in YYYYMMDD format (string or int)
      - 'Full text'             : discharge note content
      - 'Random ID'             : patient identifier
    """
    df = df_discharge.copy()

    # Parse YYYYMMDD robustly (works if the column is str or int)
    df['timestamp'] = pd.to_datetime(df['Visit Date (YYYYMMDD)'].astype(str), format='%Y%m%d', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort so concatenation is in chronological order
    df = df.sort_values(['Random ID', 'timestamp'])
    # Build entry text
    df['Discharge_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Full text'].astype(str)

    # Group per patient
    return df.groupby('Random ID', sort=False)['Discharge_Entry'].apply(' | '.join).reset_index()
def process_outpatient(df_outpatient):
    """Clean and flatten outpatient clinical notes."""
    df = df_outpatient.copy()
    
    # Filter for relevant clinical notes only
    valid_docs = ['Clinical Note', 'SGH_Consult_ExecSum_TXT']
    df = df[df['Document Item Description'].isin(valid_docs)]
    
    # Parse YYYYMMDD date format
    df['timestamp'] = pd.to_datetime(df['Visit Date (YYYYMMDD)'].astype(str), format='%Y%m%d', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort chronologically
    df = df.sort_values(['Random ID', 'timestamp'])

    # Create the string timeline
    df['Outpatient_Entry'] = (
        df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + 
        df['Document Item Description'].astype(str)
    )

    # Group into a single string per patient
    return df.groupby('Random ID', sort=False)['Outpatient_Entry'].apply(' | '.join).reset_index()

def process_endoscopy(df_endoscope):
    """Clean and flatten endoscopy reports."""
    df = df_endoscope.copy()
    
    # 1. Drop exact duplicates, NAs, and short text (< 15 chars)
    df = df.drop_duplicates()
    df = df.dropna(subset=['Summary of Procedure', 'Procedure Start Date'])
    df = df[df['Summary of Procedure'].astype(str).str.len() >= 15]

    # 2. Parse DD/MM/YYYY date safely
    df['timestamp'] = pd.to_datetime(df['Procedure Start Date'], format='%d/%m/%Y', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # 3. Sort and create entry string
    df = df.sort_values(['Random ID', 'timestamp'])
    df['Endo_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Summary of Procedure'].astype(str)

    # 4. Group per patient
    return df.groupby('Random ID', sort=False)['Endo_Entry'].apply(' | '.join).reset_index()

def create_master_patient_df(df_patients, df_labs, df_imaging, df_discharge, df_outpatient, df_endoscope):
    labs_processed = process_labs(df_labs)
    imaging_processed = process_imaging(df_imaging)
    discharge_processed = process_discharge(df_discharge)
    outpatient_processed = process_outpatient(df_outpatient)
    endo_processed = process_endoscopy(df_endoscope)

    master = (
        df_patients[['Random ID']]
        .merge(labs_processed, on='Random ID', how='left')
        .merge(imaging_processed, on='Random ID', how='left')
        .merge(discharge_processed, on='Random ID', how='left')
        .merge(outpatient_processed, on='Random ID', how='left')
        .merge(endo_processed, on='Random ID', how='left')
    )

    # ✅ STRICT FILTER: keep ONLY patients with at least ONE real source
    content_cols = [
        'Lab_Entry',
        'Img_Entry',
        'Discharge_Entry',
        'Outpatient_Entry',
        'Endo_Entry'
    ]

    master = master.dropna(subset=content_cols, how='all')

    return master.reset_index(drop=True)

# UPDATED EXECUTION
final_summary = create_master_patient_df(df_patients, df_labs, df_imaging, df_discharge, df_outpatient, df_endoscope)


empty_rows = final_summary[
    final_summary[['Lab_Entry','Img_Entry','Discharge_Entry','Outpatient_Entry','Endo_Entry']]
    .isna()
    .all(axis=1)
]

print("Rows with NO sources (should be 0):", len(empty_rows))

final_summary

In [ ]:
import pandas as pd

# 1. Load the valid IDs and convert to a set of integers for fast, accurate matching
valid_ids_df = pd.read_csv('valid_ids.csv')
valid_ids_set = set(pd.to_numeric(valid_ids_df['Random ID']).astype(int))

# 2. Convert df_truth IDs to numeric (handles floats/strings/spaces)
df_truth_numeric = pd.to_numeric(df_truth['Random ID'], errors='coerce')

# 3. Filter: Keep only rows where the ID is in our saved set
df_truth = df_truth[df_truth_numeric.fillna(-1).astype(int).isin(valid_ids_set)].copy()

# 4. Sync the format back to the clean string version (e.g., "123" not "123.0")
df_truth['Random ID'] = df_truth_numeric.loc[df_truth.index].astype(int).astype(str)

print(f"Successfully filtered df_truth. Remaining rows: {len(df_truth)}")


In [ ]:

full_mapping = {
    'Radio_Ascites_presence': 'Ascites (Y/N)',
    'Variceal_bleed_presence': 'Variceal bleed (Y/N)',
    'HE_presence': 'HE (Y/N)',
    'Spontaneous_Bacterial_Peritonitis_presence': 'SBP (Y/N)',
    'HCC_presence': 'HCC (Y/N)',
    'Portal_Vein_Thrombosis_presence': 'PVT (Y/N)',
    'TIPS_presence': 'TIPS (Y/N)',
    'Spontaneous_Bacterial_Peritonitis_presence': 'SBP (Y/N)',
    'Clinical_Ascites_presence': 'Ascites (Y/N)'
}

### View FP and FN IDs

In [ ]:
df_truth = pd.read_csv('source_of_truth.csv')
files = ['1k SBP 6.5.csv', '1k TIPS 6.5.csv','1k VARICEAL 6.5.csv','1k HE 6.5.csv','1k PVT 7.5.csv','1k HCC 7.5.csv',
         '1k CASCITES 6.5.csv',"1k RASCITES 6.5.csv"]

df_list = [pd.read_csv(f) for f in files]
final_extracted_df = pd.concat(df_list, ignore_index=True)

agg_rules = {}

for col in final_extracted_df.columns:
    if col == 'Random ID':
        continue

    if col.endswith('_presence'):
        agg_rules[col] = lambda x: int((pd.to_numeric(x, errors='coerce') == 1).any())

    elif 'date' in col.lower():
        agg_rules[col] = lambda x: next(
            (val for val in x if pd.notna(val) and str(val).strip() != 'N/A'),
            'N/A'
        )
    else:
        agg_rules[col] = 'first'

final_merged_df = (
    final_extracted_df
    .groupby('Random ID', as_index=False)
    .agg(agg_rules)
)

In [ ]:
import pandas as pd

def get_fp_fn_per_label(final_extracted_df, df_truth):
    # 1. Robust ID Cleaning
    for df in [final_extracted_df, df_truth]:
        df['Random ID'] = (
            pd.to_numeric(df['Random ID'], errors='coerce')
            .fillna(0)
            .astype(int)
            .astype(str)
        )

    # 2. Mapping
    full_mapping = {
        'Ascites_presence': 'Ascites (Y/N)',
        'Variceal_bleed_presence': 'Variceal bleed (Y/N)',
        'HE_presence': 'HE (Y/N)',
        'Spontaneous_Bacterial_Peritonitis_presence': 'SBP (Y/N)',
        'HCC_presence': 'HCC (Y/N)',
        'Portal_Vein_Thrombosis_presence': 'PVT (Y/N)',
        'TIPS_presence': 'TIPS (Y/N)',
        'SBP_presence': 'SBP (Y/N)',
        'Clinical_Ascites_presence': 'Ascites (Y/N)'
    }

    # Only process columns that exist
    active_mapping = {
        k: v for k, v in full_mapping.items()
        if k in final_extracted_df.columns
    }

    results = {}

    # 3. Per-label FP/FN computation
    for ai_col, gold_col in active_mapping.items():
        print(f"\n--- ANALYZING: {ai_col} vs {gold_col} ---")

        merged_df = pd.merge(
            final_extracted_df[['Random ID', ai_col]],
            df_truth[['Random ID', gold_col]],
            on='Random ID',
            how='inner'
        )

        # Ensure binary
        merged_df[ai_col] = pd.to_numeric(
            merged_df[ai_col], errors='coerce'
        ).fillna(0).astype(int)

        merged_df[gold_col] = pd.to_numeric(
            merged_df[gold_col], errors='coerce'
        ).fillna(0).astype(int)

        fp_ids = sorted(
            merged_df.loc[
                (merged_df[ai_col] == 1) & (merged_df[gold_col] == 0),
                'Random ID'
            ].tolist()
        )

        fn_ids = sorted(
            merged_df.loc[
                (merged_df[ai_col] == 0) & (merged_df[gold_col] == 1),
                'Random ID'
            ].tolist()
        )

        # Counts
        fp_count = len(fp_ids)
        fn_count = len(fn_ids)

        # ✅ Print what you want to see
        print(f"False Positives: {fp_count}")
        print(f"FP IDs: {fp_ids}\n")

        print(f"False Negatives: {fn_count}")
        print(f"FN IDs: {fn_ids}")
        print("=" * 50)

        # Store results for reuse
        key_base = ai_col.lower()
        results[f"{key_base}_fp_ids"] = fp_ids
        results[f"{key_base}_fn_ids"] = fn_ids
        results[f"{key_base}_fp_count"] = fp_count
        results[f"{key_base}_fn_count"] = fn_count

    return results

In [ ]:
lists = get_fp_fn_per_label(final_merged_df, df_truth)
print(lists)

### Export FN and FPs data
only exports for the selected sources I used for each hepatic event

In [ ]:
import os
import pandas as pd

# 1. Create the output folder
output_folder = "FP and FN Notes"
os.makedirs(output_folder, exist_ok=True)

# 2. Re-standardise IDs everywhere
source_dfs_list = [
    df_imaging, df_discharge, df_outpatient,
    df_endoscope, df_truth, final_merged_df
]

for df in source_dfs_list:
    df['Random ID'] = (
        pd.to_numeric(df['Random ID'], errors='coerce')
        .fillna(0)
        .astype(int)
        .astype(str)
        .str.strip()
    )

# 3. Only process labels that actually exist
active_mapping = {
    k: v for k, v in full_mapping.items()
    if k in final_merged_df.columns
}

# 4. Process each condition independently
for ai_col, gold_col in active_mapping.items():

    category = ai_col.replace('_presence', '')  # display name only

    # Align predictions vs truth
    merged_df = pd.merge(
        final_merged_df[['Random ID', ai_col]],
        df_truth[['Random ID', gold_col]],
        on='Random ID',
        how='inner'
    )

    merged_df[ai_col] = pd.to_numeric(merged_df[ai_col], errors='coerce').fillna(0).astype(int)
    merged_df[gold_col] = pd.to_numeric(merged_df[gold_col], errors='coerce').fillna(0).astype(int)

    # Identify FP / FN
    fp_ids = merged_df.loc[
        (merged_df[ai_col] == 1) & (merged_df[gold_col] == 0),
        'Random ID'
    ].unique().tolist()

    fn_ids = merged_df.loc[
        (merged_df[ai_col] == 0) & (merged_df[gold_col] == 1),
        'Random ID'
    ].unique().tolist()

    # 5. Define source targets
    if category in ['Clinical_Ascites', 'HE', 'HCC', 'Spontaneous_Bacterial_Peritonitis', 'SBP']:
        targets = [
            (df_discharge, 'Full text', 'Discharge'),
            (df_outpatient, 'Full text', 'Outpatient')
        ]
    elif category in ['Radio_Ascites', 'Portal_Vein_Thrombosis']:
        targets = [(df_imaging, 'Text', 'Imaging')]
    elif 'Variceal' in category:
        targets = [
            (df_discharge, 'Full text', 'Discharge'),
            (df_outpatient, 'Full text', 'Outpatient'),
            (df_endoscope, 'Summary of Procedure', 'Endoscopy')
        ]
    else:
        targets = [
            (df_imaging, 'Text', 'Imaging'),
            (df_discharge, 'Full text', 'Discharge'),
            (df_outpatient, 'Full text', 'Outpatient')
        ]

    # 6. Export FP and FN notes separately
    for label, ids in [("FP", fp_ids), ("FN", fn_ids)]:

        if not ids:
            print(f"No {label}s found for {category}")
            continue

        all_sources_for_label = []

        for src_df, col_name, src_label in targets:
            if col_name in src_df.columns:
                subset = (
                    src_df[src_df['Random ID'].isin(ids)]
                    [['Random ID', col_name]]
                    .copy()
                )

                if not subset.empty:
                    subset = subset.rename(columns={col_name: 'Report_Text'})
                    subset['Source_Type'] = src_label
                    all_sources_for_label.append(subset)

        if all_sources_for_label:
            final_audit_df = pd.concat(all_sources_for_label, ignore_index=True)
            final_audit_df = final_audit_df[['Random ID', 'Source_Type', 'Report_Text']]

            filename = f"{category}_{label}.csv"
            final_audit_df.to_csv(
                os.path.join(output_folder, filename),
                index=False
            )

            unique_id_count = final_audit_df['Random ID'].nunique()

            print(
                f"✅ Saved: {filename} | "
                f"Total Entries: {len(final_audit_df)} | "
                f"Unique Random IDs: {unique_id_count}"
            )

### Get FN and FP Rows as 2 Seperate dfs

In [ ]:
fp_ids_str = [str(x).strip() for x in fp_ids]
fn_ids_str = [str(x).strip() for x in fn_ids]

# 2. Clean final_summary IDs: float (123.0) -> int (123) -> string ("123")
final_summary['Random ID'] = (
    pd.to_numeric(final_summary['Random ID'], errors='coerce')
    .fillna(0)
    .astype(int)
    .astype(str)
)

# 3. Filter final_summary for both sets
fp_summary_df = final_summary[final_summary['Random ID'].isin(fp_ids_str)]
fn_summary_df = final_summary[final_summary['Random ID'].isin(fn_ids_str)]

# 4. View False Positives (AI=1, Truth=0)
print(f"--- FALSE POSITIVES: {len(fp_summary_df)} rows ---")
if not fp_summary_df.empty:
    display(fp_summary_df)
else:
    print("No False Positives found.")

print("\n" + "="*50 + "\n")

# 5. View False Negatives (AI=0, Truth=1)
print(f"--- FALSE NEGATIVES: {len(fn_summary_df)} rows ---")
if not fn_summary_df.empty:
    display(fn_summary_df)
else:
    print("No False Negatives found.")

## Filter only negatives/positives of a specific label

In [ ]:
# 1. Get the list of Random IDs where Variceal bleed is 1 in the truth table
bleed_ids = df_truth[df_truth['HCC (Y/N)'] == 1]['Random ID']

# 2. Filter final_summary to only include rows with those IDs
filtered_summary = final_summary[final_summary['Random ID'].isin(bleed_ids)]

# Optional: Reset the index if you want a clean count
filtered_summary = filtered_summary.reset_index(drop=True)
final_summary=filtered_summary
final_summary

## Misc

### Non Sleep Function
run this to keep pc on (i used this to run samples overnight)

In [ ]:
import ctypes

ES_CONTINUOUS = 0x80000000
ES_SYSTEM_REQUIRED = 0x00000001
ES_DISPLAY_REQUIRED = 0x00000002

ctypes.windll.kernel32.SetThreadExecutionState(ES_CONTINUOUS|ES_SYSTEM_REQUIRED|ES_DISPLAY_REQUIRED)

### Shutdown Funtion
shutdown pc frm jupyter notebook 

In [ ]:
import os
os.system("shutdown /s /t 1")